# Build the EpiSewer input from Covasim (shedding_hub curves already checked)

**Goal:** make everything `EpiSewer(data = ww_data, assumptions = ww_assumptions, shedding = model_shedding(load_per_case = ...))` needs.
Units: shedding is gc/mL of stool (shedding_hub label, reference event symptom onset: already checked); the load is gc/day; the concentration goes to EpiSewer in gc/mL; the flow in mL/day.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import covasim as cv

Covasim 3.1.8 (2026-05-29) — © 2020-2026 by IDM


In [ ]:
# ---- Maps (this notebook sits in the same folder as run_covasim.py) ------------------------------------------------
REPO_ROOT                 = Path.cwd().parents[1]
SAN_DIEGO_DATA_FOLDER     = REPO_ROOT / 'docs' / 'tutorials' / 'san-diego-data'
POPULATION_DENSITY_RASTER = SAN_DIEGO_DATA_FOLDER / 'san_diego_county_ppp_2020.tif'
ZIP_CODE_BOUNDARIES       = SAN_DIEGO_DATA_FOLDER / 'sandiego_zip_boundaries.geojson'
SEWERSHED_BOUNDARIES      = SAN_DIEGO_DATA_FOLDER / 'sandiego_sewersheds.geojson'

OUTPUT_FOLDER = Path('episewer_input')         # the CSVs for the R script are written here
OUTPUT_FOLDER.mkdir(exist_ok=True)

# Overview (shedding_hub curves already checked)

**Goal:** make everything `EpiSewer(data = ww_data, assumptions = ww_assumptions, shedding = model_shedding(load_per_case = ...))` needs.
Units: shedding is gc/mL of stool (shedding_hub label, reference event symptom onset: already checked); the load is gc/day; the concentration goes to EpiSewer in gc/mL; the flow in mL/day.

## Settings
Epidemic: `N_AGENTS=50000`, `N_DAYS=180`, `TRANSMISSION_PROB_PER_CONTACT=0.012`, `N_INITIAL_INFECTIONS=10`, `FRIEND_COUNT_DISPERSION=None`, `INFECTIOUSNESS_DISPERSION=0.45`, `DAILY_TEST_PROB_IF_SYMPTOMATIC=0.1` (not used by EpiSewer; kept so the run matches run_covasim.py).
Maps for `pop_type='rank'`: `POPULATION_DENSITY_RASTER`, `ZIP_CODE_BOUNDARIES`, `SEWERSHED_BOUNDARIES` (same three files as run_covasim.py).

## Our assumptions
- `STOOL_ML_PER_PERSON_PER_DAY=128`: median fecal wet mass of 128 g per person per day, with 1 g taken as 1 mL [1]
- `SEWAGE_FLOW_LITRES_PER_PERSON_PER_DAY=270`: Point Loma treats 136 to 175 million gallons per day for about 2.2 million residents [2].
  - Calculation: 1 million gallons = 3.785 million L, so 136 MGD = 5.1 × 10⁸ L and 175 MGD = 6.6 × 10⁸ L per day.
  - Divided by 2.2 million residents this is 234 to 301 L per person per day. We use 270, the midpoint, rounded.
  - It is a per-resident average: the plant flow also includes commercial, industrial and infiltration water.
- `LAB_NOISE_CV=0.2`: placeholder lab noise (multiplicative, mean 1); no reference

## Technical settings
- `SHEDDING_PROFILE_DAYS=40`: days after infection kept when building the shedding distribution (check that the window holds about 100% of the shedding)
- A sample is taken every day (the best case)

## Steps
1. **Run Covasim** with the settings above. Copy `sim.shedding_lookup` after `initialize()` and before `run()`. Take `viral_shedding_hist` (agents × calendar days, gc/mL), `date_exposed`, `date_infectious` and `people.sewershed`.
2. **Check:** sliding each agent's curve (`shedding_lookup` row) to start at `date_infectious` reproduces `viral_shedding_hist`.
3. **Site load:** add up the agents (gc/mL), then × `STOOL_ML_PER_PERSON_PER_DAY` (128 mL), giving `load_gc_per_day`.
4. **Flow and concentration:** flow = agents × `SEWAGE_FLOW_LITRES_PER_PERSON_PER_DAY` (270 L). Concentration = load ÷ flow, in gc/L.
5. **Measurement:** concentration × lab noise (mean 1, CV 0.2). Leave out days with a concentration of 0.
6. **EpiSewer data tables:** concentration in gc/mL (÷ 1000) and flow in mL/day (× 1000). These are `ww_data`.
7. **Shedding distribution:** line each agent's shedding up by days since `date_exposed`, average over agents, and divide by the total (sums to 1). This is `shedding_dist`.
8. **Load per case:** mean total shedding of one infection × 128 mL, in gc per infection. This goes to `load_per_case_assume`.
9. **Generation time:** from `infection_log`, the infectee's infection day minus the infector's, from day 1. This is `generation_dist`.
10. **Save the CSVs** for the R script. For judging only: true infections per day from `infection_log`.

## What each EpiSewer argument comes from
| EpiSewer argument | From step |
|---|---|
| `data` (`ww_data`: concentration, flow) | 3–6 |
| `shedding_dist`, `shedding_reference = "infection"` | 7 |
| `load_per_case` | 8 |
| `generation_dist` | 9 |
| not given to EpiSewer: true infections | 10 |

## References
1. Rose C, Parker A, Jefferson B, Cartmell E. The characterization of feces and urine: a review of the literature to inform advanced treatment technology. *Crit Rev Environ Sci Technol* 2015; 45:1827–1879. doi:10.1080/10643389.2014.1000761
2. City of San Diego. Point Loma Wastewater Treatment Plant fact sheet (about 175 MGD, more than 2.2 million residents); Point Loma annual reports 2014 (139.2 MGD) and 2016 (136.1 MGD).

## Epidemic settings

In [3]:
# ---- The epidemic -------------------------------------------------------------------------------------------------
N_AGENTS                  = 50000      # number of simulated people
N_DAYS                    = 180        # length of the simulated epidemic, in days
TRANSMISSION_PROB_PER_CONTACT = 0.012  # Covasim's `beta`: chance of transmission per infectious contact (Covasim default is 0.016)
N_INITIAL_INFECTIONS      = 10         # agents infected on day 0 (the "seeds": infected by construction, not by anyone)
FRIEND_COUNT_DISPERSION   = None       # "k1": None = Poisson (the default), so it is not passed to Covasim below
INFECTIOUSNESS_DISPERSION = 0.45       # "k2": how uneven infectiousness is between people (superspreading)
DAILY_TEST_PROB_IF_SYMPTOMATIC = 0.1   # not used by EpiSewer; kept so the run matches run_covasim.py
SEED                      = 1

## Assumptions

In [4]:
# ---- Our assumptions and technical setting ---------------------------------------------------------------------------
STOOL_ML_PER_PERSON_PER_DAY           = 128    # Ref: Rose et al. 2015
SEWAGE_FLOW_LITRES_PER_PERSON_PER_DAY = 270    # Ref: Point Loma, City of San Diego
LAB_NOISE_CV                          = 0.2    # placeholder
SHEDDING_PROFILE_DAYS                 = 40

## Step 1. Run Covasim

In [5]:
def run_covasim(seed):

    testing = cv.test_prob(symp_prob=DAILY_TEST_PROB_IF_SYMPTOMATIC, asymp_prob=0.0, start_day=0)
    
    sim = cv.Sim(pop_type='rank', 
                 pop_size=N_AGENTS, 
                 pop_infected=N_INITIAL_INFECTIONS,
                 n_imports=0,
                 beta=TRANSMISSION_PROB_PER_CONTACT,
                 n_days=N_DAYS, rand_seed=seed, verbose=0,
                 beta_dist=dict(dist='neg_binomial', par1=1.0, par2=INFECTIOUSNESS_DISPERSION, step=0.01),
                 interventions=testing, 
                 density_raster=str(POPULATION_DENSITY_RASTER),
                 boundary_shapefile=str(ZIP_CODE_BOUNDARIES), 
                 sewershed_file=str(SEWERSHED_BOUNDARIES))
    
    sim.initialize(use_workers=True)
    shedding_lookup = np.array(sim.shedding_lookup).copy()     # copy now: Covasim redraws it after the run
    sim.run()
    return sim, shedding_lookup

sim, shedding_lookup = run_covasim(SEED)

/storage/hcraddock/GitHub/covasim4wastewater/covasim/population.py:67: RuntimeWarning: 
The boundary file has no "population" column, so regions are weighted by their area. Add a "population" column to weight by population (it is not needed when pop_type="rank" uses a density_raster).
  cvm.warn(


In [6]:
viral_shedding_hist = np.asarray(sim.results['viral_shedding_hist'], dtype=float)   # agents x days, gc per mL of stool

date_exposed        = np.asarray(sim.people.date_exposed, dtype=float)               # day of infection (NaN = never)

date_infectious     = np.asarray(sim.people.date_infectious, dtype=float)            # day infectious (NaN = never)

sewershed_id        = np.asarray(sim.people.sewershed)                               # catchment of each agent

infection_log       = sim.people.infection_log                                       # every transmission: source, target, date

dates               = pd.to_datetime(sim.datevec)

n_days              = len(dates)

print('viral_shedding_hist:', viral_shedding_hist.shape, '| shedding_lookup:', shedding_lookup.shape, '| infections:', len(infection_log))

viral_shedding_hist: (50000, 181) | shedding_lookup: (50000, 180) | infections: 27217


## Step 2. Check: each agent's curve starts at `date_infectious`

In [7]:
infectious_agents = np.where(~np.isnan(date_infectious))[0][:2000]      # a sample is enough

def rebuild_row(agent, shift):
    # slide the agent's curve so that its first value lands on date_infectious + shift
    start = int(date_infectious[agent]) + shift
    length = max(0, min(shedding_lookup.shape[1], n_days - start))
    row = np.zeros(n_days)
    row[start:start + length] = shedding_lookup[agent, :length]
    return row

def matches_early(agent, shift, days=15):
    first = int(date_infectious[agent])
    row = rebuild_row(agent, shift)
    return np.allclose(row[first:first + days], viral_shedding_hist[agent, first:first + days], rtol=1e-4, atol=1e-3)

for shift in (0, 1):
    matches = [matches_early(a, shift) for a in infectious_agents]
    print(f'curve starts at date_infectious + {shift}: matches viral_shedding_hist for {sum(matches)} of {len(matches)} agents')

curve starts at date_infectious + 0: matches viral_shedding_hist for 2000 of 2000 agents
curve starts at date_infectious + 1: matches viral_shedding_hist for 54 of 2000 agents


## Step 3. Site load: add up the agents, then × 128 mL of stool

In [8]:
sites = {'ALL': np.ones(N_AGENTS, dtype=bool)}
for sid, name in sim.sewershed_labels.items():           # {integer id: plant name}
    sites[name] = (sewershed_id == sid)

def site_load(mask):
    return viral_shedding_hist[mask].sum(axis=0) * STOOL_ML_PER_PERSON_PER_DAY     # gc/mL x mL = gc/day

print({name: int(mask.sum()) for name, mask in sites.items()}, '(agents per site)')
print(f"ALL: peak load = {site_load(sites['ALL']).max():,.3e} gc/day")

{'ALL': 50000, 'Encina': 9155, 'Point Loma': 33862, 'South Bay': 2213} (agents per site)
ALL: peak load = 1.569e+10 gc/day


## Step 4. Flow and concentration (gc/L)

In [ ]:
def site_flow(mask):
    return mask.sum() * SEWAGE_FLOW_LITRES_PER_PERSON_PER_DAY                      # L/day

all_load, all_flow = site_load(sites['ALL']), site_flow(sites['ALL'])
print(f'ALL: flow = {all_flow:,} L/day | peak concentration = {(all_load / all_flow).max():,.1f} gc/L (no noise)')

## Step 5. Measurement: concentration × lab noise (mean 1, CV 0.2)

In [ ]:
rng = np.random.default_rng(SEED)
sigma = np.sqrt(np.log(1 + LAB_NOISE_CV ** 2))

def add_lab_noise(concentration):
    return concentration * rng.lognormal(mean=-sigma ** 2 / 2, sigma=sigma, size=n_days)      # multiplier has mean 1, CV 0.2

## Step 6. EpiSewer data: concentration (gc/mL) and flow (mL/day)

In [ ]:
infected_agent = np.array([event['target'] for event in infection_log]).astype(int)
infection_day  = np.array([event['date'] for event in infection_log]).astype(int)

def site_table(name, mask):
    load, flow = site_load(mask), site_flow(mask)
    concentration = load / flow                                                   # gc/L
    return pd.DataFrame({
        'site': name, 'date': dates, 'n_people': mask.sum(),
        'true_infections': np.bincount(infection_day[mask[infected_agent]], minlength=n_days)[:n_days],   # the truth, only to judge
        'load_gc_per_day': load, 'flow_L_per_day': flow,
        'concentration_true': concentration, 'concentration_noisy': add_lab_noise(concentration)})

ww_daily = pd.concat([site_table(name, mask) for name, mask in sites.items()], ignore_index=True)

kept = ww_daily[ww_daily['concentration_noisy'] > 0]                              # leave out days with concentration 0
ww_measurements = pd.DataFrame({'site': kept['site'], 'date': kept['date'], 'concentration': kept['concentration_noisy'] / 1000})   # gc/L -> gc/mL
ww_flows = pd.DataFrame({'site': ww_daily['site'], 'date': ww_daily['date'], 'flow': ww_daily['flow_L_per_day'] * 1000})             # L/day -> mL/day

ww_measurements.head(3), ww_flows.head(3)

## Step 7. Shedding distribution: share of one infection's shedding on each day after `date_exposed`

In [ ]:
n_infections_per_agent = np.bincount(infected_agent, minlength=N_AGENTS)
used_agents = np.where((date_exposed <= n_days - SHEDDING_PROFILE_DAYS) & (n_infections_per_agent == 1))[0]   # infected once, early enough

def shedding_by_days_since_infection(agents):
    return np.array([viral_shedding_hist[a, int(date_exposed[a]):int(date_exposed[a]) + SHEDDING_PROFILE_DAYS] for a in agents]).sum(axis=0)

shedding_sum = shedding_by_days_since_infection(used_agents)                       # gc/mL summed over agents, day 0, 1, 2 ... after infection
shedding_dist = shedding_sum / shedding_sum.sum()                                  # sums to 1, no unit

print(f'{len(used_agents):,} agents used | window holds {shedding_sum.sum() / viral_shedding_hist[used_agents].sum():.1%} of their shedding | sums to {shedding_dist.sum():.3f}')

## Step 8. Load per case: total shedding of one infection × 128 mL of stool

In [ ]:
load_per_case = shedding_sum.sum() / len(used_agents) * STOOL_ML_PER_PERSON_PER_DAY        # gene copies per infection
print(f'load_per_case = {load_per_case:,.3e} gene copies per infection')

## Step 9. Generation time: infectee's infection day minus the infector's

In [ ]:
def generation_times(infection_log):
    log = pd.DataFrame(infection_log)[['source', 'target', 'date']]
    events = log.dropna(subset=['source']).astype({'source': int}).sort_values('date')                    # drop the seeds
    infector_infections = log[['target', 'date']].rename(columns={'target': 'source', 'date': 'source_infection_day'}).sort_values('source_infection_day')
    # for each transmission, the infector's latest infection on or before that day
    merged = pd.merge_asof(events, infector_infections, left_on='date', right_on='source_infection_day', by='source', direction='backward')
    merged['generation_time'] = merged['date'] - merged['source_infection_day']
    return merged

transmissions = generation_times(infection_log)
counts = np.bincount(transmissions['generation_time'].dropna().astype(int))
generation_dist = counts[1:] / counts[1:].sum()                                    # days 1, 2, 3 ... sums to 1

print(f"{len(transmissions):,} transmissions | mean generation time {transmissions['generation_time'].mean():.2f} days | days 1 to {len(generation_dist)}")

## Step 10. Save the CSVs for the R script

In [ ]:
def save(table, name):
    table.to_csv(OUTPUT_FOLDER / name, index=False)

save(ww_measurements, 'ww_data_measurements.csv')       # site, date, concentration (gc/mL)  -> ww_data
save(ww_flows, 'ww_data_flows.csv')                     # site, date, flow (mL/day)          -> ww_data
save(pd.DataFrame({'day': np.arange(SHEDDING_PROFILE_DAYS), 'pmf': shedding_dist}), 'covasim_ww_shedding_dist.csv')
save(transmissions, 'covasim_transmission.csv')         # R builds generation_dist from generation_time
save(ww_daily, 'covasim_ww_daily.csv')                  # the truth, only used to judge the result
save(pd.DataFrame({'date': dates, 'r_eff': np.array(sim.results['r_eff'].values)}), 'covasim_cases.csv')    # r_eff, only to judge Rt
save(pd.DataFrame({'seed': [SEED], 'LOAD_PER_CASE': [load_per_case], 'WW_STOOL_ML_PER_DAY': [STOOL_ML_PER_PERSON_PER_DAY],
                   'WW_FLOW_PER_PERSON': [SEWAGE_FLOW_LITRES_PER_PERSON_PER_DAY], 'WW_NOISE_CV': [LAB_NOISE_CV]}), 'covasim_run_info.csv')

print(sorted(path.name for path in OUTPUT_FOLDER.glob('*.csv')))